# Two-SG fusion methodology: run, diagnostics, posteriors, RUL

One full run of the hybrid data-fusion methodology per steam generator (**SG A**,
**SG B**): every CC/PC cycle assimilated on its full field data (no observation-size
sweep), reporting at each step which input variable was calibrated, saving the posterior
parameter distributions and the BMU / BMU+EnKS trajectories, plotting prior/posterior
trajectories and RUL distributions on the prognostics window, and finally emitting the
RUL statistics tables (Mean, Std, $q_{05}$, Median, $q_{95}$).

Renamed and rebuilt from `rul_two_sg_tables.ipynb`: that notebook only ever surfaced the
final `RULStats` (via `run_observation_size(pipe, k=K_ALL, ...)`); this one calls the new
`run_full_methodology()` instead, which returns every cycle's `CycleResult` (active
variable, convergence, posterior marginals) alongside the `PrognosticsStages` trajectory
ensembles -- everything below is built from those two objects.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import numpy as nps
import pandas as pd
from scipy.stats import gaussian_kde

from clogging_prognostics import CloggingPipeline, PipelineConfig, run_full_methodology

RESULTS_DIR = 'results'
POSTERIORS_DIR = os.path.join(RESULTS_DIR, 'posteriors')
TRAJECTORIES_DIR = os.path.join(RESULTS_DIR, 'trajectories')
FIGURES_DIR = os.path.join(RESULTS_DIR, 'figures')
for d in (POSTERIORS_DIR, TRAJECTORIES_DIR, FIGURES_DIR):
    os.makedirs(d, exist_ok=True)

## Per-SG configuration

The two SGs differ in cleaning history, so `n_cycles` and the window anchors differ.
Set these from each SG's metadata (cleaning times, in **hours**, become the split anchors;
`sim_time` is in the raw grid unit = metadata-hours / 24).

No more `K_ALL` workaround: `run_full_methodology` always assimilates every cycle on its
full, untruncated field data, so there is no observation-size truncation to clamp.

In [ ]:
BASE_DIRECTORY = 'DATA/'   # dir holding <SG>_MC_UNIFORM_HL/CL.npy, <SG>_TIME.npy,
                                    # <SG>_METADATA.json, <SG>_FIELD_DATA.csv
THRESHOLDS = {'SG A': None, 'SG B': None}   # tau_c^*  (%) per steam generator
N_RUL_SAMPLES = 1000

SG_CONFIGS = {
    'SG A': dict(
        sg_tag='SGA',                     # DATA/ file prefix and output filename tag
        n_cycles=3,                       # two cleanings -> three windows
        split_sim_hours=221938,           # last cleaning (2nd) -> start of prognostics window
        prognostics_split_hours=221938,   # first grid anchor inside the last window
    ),
    'SG B': dict(
        sg_tag='SGB',
        n_cycles=2,                       # one cleaning -> two windows
        split_sim_hours=176580,
        prognostics_split_hours=176580,
    ),
}

## Run both SGs

`run_full_methodology` calls `pipe.run()` (which prints, as it goes, which variable is
active and whether each cycle converged) and then builds the prior/BMU/BMU+EnKS trajectory
ensembles via `compute_prognostics_stages`.

> **Note on the BMU trajectory for SG A**: `compute_prognostics_stages` splices the
> concatenated BMU trajectory from exactly the *first two* cycles (one-cleaning
> assumption, inherited unchanged from the original notebook). SG A has **three**
> windows (two cleanings); its third cycle's diagnostics and posterior distribution are
> still computed and saved below, but the BMU / BMU+EnKS trajectory and RUL figures do
> not reflect that third window. Generalizing the splice to arbitrary cycle counts is a
> separate follow-up, not attempted here.

In [ ]:
runs = {}
for sg_name, cfg_dict in SG_CONFIGS.items():
    print(f'=== {sg_name} ===')
    cfg = PipelineConfig(
        base_directory=BASE_DIRECTORY,
        input_dimension=7,
        n_cycles=cfg_dict['n_cycles'],
        clogging_threshold=THRESHOLDS[sg_name],
    )
    pipe = CloggingPipeline(cfg).setup(sg_tag=cfg_dict['sg_tag'])
    cycle_results, stages = run_full_methodology(
        pipe,
        split_sim_hours=cfg_dict['split_sim_hours'],
        prognostics_split_hours=cfg_dict['prognostics_split_hours'],
        n_rul_samples=N_RUL_SAMPLES,
    )
    runs[sg_name] = dict(cfg=cfg, pipe=pipe, cycle_results=cycle_results, stages=stages)
    print()

## Assimilation diagnostics: which variable was calibrated at each step

One row per cycle (window): the HSIC-selected active variable, whether the
refine-until-converged loop converged, how many inner iterations it took, and the final
KL(posterior $\Vert$ prior) that triggered the stop.

In [ ]:
def diagnostics_frame(cfg, cycle_results):
    rows = []
    for res in cycle_results:
        rows.append(dict(
            Cycle=res.cycle_index,
            **{'Active variable': cfg.prior.names[res.active_index]},
            **{'Active index': res.active_index},
            Converged=res.converged,
            **{'Iterations': res.n_iterations},
            **{'Final KL': res.kl_history[-1] if res.kl_history else float('nan')},
        ))
    return pd.DataFrame(rows)

diagnostics = {}
for sg_name, run in runs.items():
    print(sg_name)
    df = diagnostics_frame(run['cfg'], run['cycle_results'])
    diagnostics[sg_name] = df
    display(df)

## Save posterior parameter distributions (one file per window per SG)

Each cycle's refined marginals (only the HSIC-active variable actually moves away from
its prior; the rest stay at their cycle-local prior), sampled to `N_RUL_SAMPLES` draws and
written to `results/posteriors/<SG>_cycle<i>_posterior.csv`.

In [ ]:
def save_posterior_samples(sg_tag, cfg, cycle_results, n_samples=N_RUL_SAMPLES):
    paths = []
    for res in cycle_results:
        samples = np.asarray([
            np.asarray(m.getSample(n_samples)).reshape(-1) for m in res.marginals
        ]).T
        df = pd.DataFrame(samples, columns=list(cfg.prior.names))
        path = os.path.join(POSTERIORS_DIR, f'{sg_tag}_cycle{res.cycle_index}_posterior.csv')
        df.to_csv(path, index=False)
        paths.append(path)
    return paths

for sg_name, run in runs.items():
    paths = save_posterior_samples(SG_CONFIGS[sg_name]['sg_tag'], run['cfg'], run['cycle_results'])
    for p in paths:
        print('wrote', p)

## Save the BMU / BMU+EnKS posterior trajectories

The prior/BMU/BMU+EnKS trajectory ensembles on the prognostics window, plus their time
axes, written to `results/trajectories/<SG>_<stage>.npy` / `<SG>_<stage>_time.npy`.

In [ ]:
def save_trajectories(sg_tag, stages):
    arrays = {
        'prior': (stages.prior_traj, stages.pc_time),
        'bmu': (stages.bmu_traj, stages.pc_time),
        'bmu_enks': (stages.bmu_enks_traj, stages.enks_time),
    }
    for stage, (traj, t) in arrays.items():
        traj_path = os.path.join(TRAJECTORIES_DIR, f'{sg_tag}_{stage}.npy')
        time_path = os.path.join(TRAJECTORIES_DIR, f'{sg_tag}_{stage}_time.npy')
        np.save(traj_path, traj)
        np.save(time_path, np.asarray(t))
        print('wrote', traj_path, traj.shape, '/', time_path, np.asarray(t).shape)

for sg_name, run in runs.items():
    save_trajectories(SG_CONFIGS[sg_name]['sg_tag'], run['stages'])

## Load the saved trajectories

Reads back the `.npy` files written above -- one `(n_ensemble, n_time)` trajectory array
plus its `(n_time,)` time axis (days) per stage per SG. Only needs `SG_CONFIGS` and
`TRAJECTORIES_DIR` (set in the first two cells), so this cell works on its own in a fresh
kernel too: skip re-running the (expensive) assimilation above and just reload a previous
run's trajectories, e.g. for further analysis or re-plotting.

In [ ]:
def load_trajectories(sg_tag):
    """Load one SG's saved prior/BMU/BMU+EnKS trajectories + time axes.

    Returns ``{stage: (traj, time)}`` for stage in {'prior', 'bmu', 'bmu_enks'}.
    """
    trajectories = {}
    for stage in ('prior', 'bmu', 'bmu_enks'):
        traj_path = os.path.join(TRAJECTORIES_DIR, f'{sg_tag}_{stage}.npy')
        time_path = os.path.join(TRAJECTORIES_DIR, f'{sg_tag}_{stage}_time.npy')
        trajectories[stage] = (np.load(traj_path), np.load(time_path))
    return trajectories

loaded_trajectories = {}
for sg_name, cfg_dict in SG_CONFIGS.items():
    loaded_trajectories[sg_name] = load_trajectories(cfg_dict['sg_tag'])
    for stage, (traj, t) in loaded_trajectories[sg_name].items():
        print(f'{sg_name:10s} {stage:9s} traj {traj.shape}  time {t.shape} '
              f'[{t.min():.0f}, {t.max():.0f}] d')

## Plot: prior/posterior trajectories and RUL distributions on the prognostics window

One 2x3 figure per SG, columns = Prior / BMU / BMU+EnKS:
- **top row**: RUL histogram (+ KDE) for a crossing threshold of $\tau_c^\ast=$
  the per-SG threshold (X % for SG A, X % for SG B), cumulative across stages (the BMU panel overlays Prior+BMU, the
  BMU+EnKS panel overlays all three) -- shows how the RUL estimate sharpens/shifts as
  BMU then EnKS are added;
- **bottom row**: that stage's own trajectory ensemble on the prognostics window, with
  the threshold as a dashed line, the last-cycle field data overlaid (ESTICOL/ETV, colored
  as in `visualize_clogging_data.ipynb`), and a vertical line at the last available field
  observation (after which the trajectories are pure extrapolation).

In [ ]:
STAGE_ORDER = ['prior', 'bmu', 'bmu_enks']
STAGE_LABEL = {'prior': 'Prior', 'bmu': 'BMU', 'bmu_enks': 'BMU+EnKS'}
STAGE_COLOR = {'prior': 'green', 'bmu': 'royalblue', 'bmu_enks': 'lightcoral'}


def _rul_by_stage(stages, threshold):
    stage_defs = {
        'prior': (stages.prior_traj, stages.pc_time),
        'bmu': (stages.bmu_traj, stages.pc_time),
        'bmu_enks': (stages.bmu_enks_traj, stages.enks_time),
    }
    rul = {}
    for stage, (traj, t) in stage_defs.items():
        crossings = []
        for i in range(traj.shape[0]):
            idx = np.where(traj[i] > threshold)[0]
            if idx.size:
                crossings.append(t[idx[0]] - t[0])
        rul[stage] = np.asarray(crossings)
    return rul, stage_defs


def plot_prognostics_summary(stages, dataset, threshold, sg_title):
    rul_by_stage, stage_defs = _rul_by_stage(stages, threshold)

    last_channels = dataset.data_with_time_division[-1]
    field_esticol = np.asarray(last_channels[0])
    field_etv = np.asarray(last_channels[1])
    last_times = [c[:, 0].max() for c in (field_esticol, field_etv) if len(c)]
    last_field_time = max(last_times) if last_times else None

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    for col, stage in enumerate(STAGE_ORDER):
        traj, t = stage_defs[stage]

        # top row: cumulative RUL histogram + KDE, stages up to and including this one
        ax = axes[0, col]
        for s2 in STAGE_ORDER[:col + 1]:
            rul = rul_by_stage[s2]
            if rul.size == 0:
                continue
            counts, bins, _ = ax.hist(
                rul, bins=10, color=STAGE_COLOR[s2], alpha=0.6,
                edgecolor='k', label=f'{STAGE_LABEL[s2]} RUL',
            )
            if rul.size > 1 and np.std(rul) > 0:
                kde = gaussian_kde(rul)
                xs = np.linspace(bins[0], bins[-1], 200)
                bin_width = bins[1] - bins[0]
                ax.plot(xs, kde(xs) * rul.size * bin_width, color=STAGE_COLOR[s2])
        ax.set_ylabel('Frequency')
        if col > 0:
            ax.legend(fontsize=8)

        # bottom row: this stage's own trajectory ensemble
        axb = axes[1, col]
        axb.axvspan(t.min(), t.max(), color='lightgray', zorder=0)
        for i in range(traj.shape[0]):
            axb.plot(t, traj[i], color=STAGE_COLOR[stage], alpha=0.15, linewidth=0.8)
        axb.axhline(threshold, color='darkred', linestyle='--', linewidth=1.5,
                    label=r'$\tau_c^\ast$' if col == 0 else None)
        if last_field_time is not None:
            axb.axvline(last_field_time, color='orange', linewidth=2,
                        label='Last field obs.' if col == 0 else None)
        if len(field_esticol):
            axb.scatter(field_esticol[:, 0], field_esticol[:, 1], c='lime', marker='o',
                        edgecolors='k', linewidths=0.4, s=16, zorder=11,
                        label='ESTICOL' if col == 0 else None)
        if len(field_etv):
            axb.scatter(field_etv[:, 0], field_etv[:, 1], c='magenta', marker='^',
                        edgecolors='k', linewidths=0.4, s=22, zorder=12,
                        label='ETV' if col == 0 else None)
        axb.set_xlabel(r'$t\;(d)$', fontsize=13)
        if col == 0:
            axb.set_ylabel(r'$\tau_c$', fontsize=13)
            axb.legend(fontsize=8, loc='lower right')
        axb.set_title(STAGE_LABEL[stage])

    fig.suptitle(sg_title)
    fig.tight_layout()
    return fig


for sg_name, run in runs.items():
    sg_tag = SG_CONFIGS[sg_name]['sg_tag']
    fig = plot_prognostics_summary(run['stages'], run['pipe'].dataset, THRESHOLDS[sg_name], sg_name)
    fig_path = os.path.join(FIGURES_DIR, f'{sg_tag}_prognostics_summary.png')
    fig.savefig(fig_path, dpi=200)
    print('wrote', fig_path)
    plt.show()

## RUL statistics per stage

In [ ]:
STAGE_LABEL_TEX = {'prior': 'Prior', 'bmu': 'BMU', 'bmu_enks': r'BMU\,+\,EnKS'}

for sg_name, run in runs.items():
    dataset = run['pipe'].dataset
    n_obs = len(dataset.data_with_time_division[-1][0])   # ESTICOL count, last cycle
    run['rul_stats'] = run['stages'].rul_stats(THRESHOLDS[sg_name], n_obs)

results = {sg_name: run['rul_stats'] for sg_name, run in runs.items()}

## On-screen tables

In [ ]:
def stats_frame(by_stage):
    rows = []
    for st in STAGE_ORDER:
        s = by_stage[st]
        rows.append(dict(Stage=STAGE_LABEL[st], Mean=s.mean, Std=s.std, q05=s.q05,
                         Median=s.median, q95=s.q95))
    return pd.DataFrame(rows)[['Stage','Mean','Std','q05','Median','q95']]

for sg_name in results:
    print(sg_name)
    display(stats_frame(results[sg_name]).round(1))

## LaTeX tables (paste-ready, matching the manuscript)

Emits one `table` float per SG with the exact columns Stage / Mean / Std / $q_{05}$ / Median / $q_{95}$.
Edit the caption threshold/units to match your text.

In [ ]:
def latex_table(by_stage, sg_display, label, threshold, unit='days', dp=1):
    def f(x):
        return '--' if (x != x) else f'{x:.{dp}f}'   # NaN -> --
    lines = []
    lines.append(r'\begin{table}[ht]')
    lines.append(r'  \centering')
    lines.append(rf'  \caption{{RUL statistics on the prognostics window for a specified '
                 rf'threshold $\tau_c^\ast={threshold:g}\%$, at each stage of the methodology '
                 rf'for \textbf{{{sg_display}}}. Values in {unit}.}}')
    lines.append(rf'  \label{{{label}}}')
    lines.append(r'  \begin{tabular}{lrrrrr}')
    lines.append(r'    \toprule')
    lines.append(r'    Stage & Mean & Std & $q_{05}$ & Median & $q_{95}$ \\')
    lines.append(r'    \midrule')
    for st in STAGE_ORDER:
        s = by_stage[st]
        lines.append(f'    {STAGE_LABEL_TEX[st]} & {f(s.mean)} & {f(s.std)} & '
                     f'{f(s.q05)} & {f(s.median)} & {f(s.q95)} \\\\')
    lines.append(r'    \bottomrule')
    lines.append(r'  \end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

LABELS = {'SG A': 'tab:rul_sga', 'SG B': 'tab:rul_sgb'}
for sg_name in results:
    print(latex_table(results[sg_name], sg_name, LABELS[sg_name], THRESHOLDS[sg_name]))
    print()

## (Optional) write the LaTeX to a file

In [ ]:
with open('rul_tables.tex', 'w') as fh:
    for sg_name in results:
        fh.write(latex_table(results[sg_name], sg_name, LABELS[sg_name], THRESHOLDS[sg_name]))
        fh.write('\n\n')
print('wrote rul_tables.tex')